# Notebook 02: Exploratory Data Analysis
## Purpose: Understand sensor behavior, identify key features, document data quality
## Input: workspace.predictive_maintenance.bronze_nasa_sensor_raw
## Output: Sensor selection decision, data quality report, degradation patterns documented

In [0]:
# Load Bronze table
df = spark.table("workspace.predictive_maintenance.bronze_nasa_sensor_raw")

print(f"Total rows:    {df.count():,}")
print(f"Total columns: {len(df.columns)}")
print(f"Datasets:      {df.select('source_dataset').distinct().collect()}")
df.printSchema()

In [0]:
import pandas as pd

# Convert to pandas for EDA (small enough)
pdf = df.toPandas()

print("=== NULL COUNTS ===")
nulls = pdf.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else " No nulls found")

print("\n=== BASIC STATS ===")
print(f"Unique machines:     {pdf['unit_id'].nunique()}")
print(f"Max cycles (FD001):  {pdf[pdf['source_dataset']=='FD001']['cycle'].max()}")
print(f"Max cycles (FD002):  {pdf[pdf['source_dataset']=='FD002']['cycle'].max()}")
print(f"Avg cycles per machine: {pdf.groupby('unit_id')['cycle'].max().mean():.1f}")

In [0]:
import matplotlib.pyplot as plt
import numpy as np

sensor_cols = [f"sensor_{i}" for i in range(1, 22)]

# Calculate variance for each sensor
variances = pdf[sensor_cols].var().sort_values(ascending=False)

# Plot
fig, ax = plt.subplots(figsize=(14, 5))
colors = ['#1565C0' if v > variances.median() else '#90CAF9' for v in variances]
ax.bar(variances.index, variances.values, color=colors)
ax.set_title("Sensor Variance — High Variance = More Useful for Prediction", 
             fontsize=14, fontweight='bold')
ax.set_xlabel("Sensor")
ax.set_ylabel("Variance")
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig("/Volumes/workspace/predictive_maintenance/raw_data/chart1_sensor_variance.png")
plt.show()

# Print top 7 sensors
top_sensors = variances.head(7).index.tolist()
print(f"\nTop 7 high-variance sensors: {top_sensors}")
print(f"Low-variance sensors to DROP: {variances.tail(14).index.tolist()}")

In [0]:
# Calculate RUL for each machine
max_cycles = pdf.groupby('unit_id')['cycle'].max().reset_index()
max_cycles.columns = ['unit_id', 'max_cycle']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# FD001
fd001_life = pdf[pdf['source_dataset']=='FD001'].groupby('unit_id')['cycle'].max()
axes[0].hist(fd001_life, bins=20, color='#1565C0', edgecolor='white', alpha=0.85)
axes[0].set_title(f"FD001 — Machine Lifetime Distribution\nAvg: {fd001_life.mean():.0f} cycles", 
                   fontweight='bold')
axes[0].set_xlabel("Total Cycles Before Failure")
axes[0].set_ylabel("Number of Machines")

# FD002  
fd002_life = pdf[pdf['source_dataset']=='FD002'].groupby('unit_id')['cycle'].max()
axes[1].hist(fd002_life, bins=20, color='#283593', edgecolor='white', alpha=0.85)
axes[1].set_title(f"FD002 — Machine Lifetime Distribution\nAvg: {fd002_life.mean():.0f} cycles",
                   fontweight='bold')
axes[1].set_xlabel("Total Cycles Before Failure")

plt.tight_layout()
plt.savefig("/Volumes/workspace/predictive_maintenance/raw_data/chart2_rul_distribution.png")
plt.show()
print(f"FD001 avg machine life: {fd001_life.mean():.0f} cycles")
print(f"FD002 avg machine life: {fd002_life.mean():.0f} cycles")

In [0]:
# Pick 3 sample machines from FD001
sample_machines = [1, 2, 3]
fd001_pdf = pdf[pdf['source_dataset'] == 'FD001']

sensors_to_plot = ['sensor_2', 'sensor_7', 'sensor_11']

fig, axes = plt.subplots(3, 3, figsize=(16, 10))
fig.suptitle("Sensor Degradation Over Time — 3 Sample Machines", 
             fontsize=14, fontweight='bold')

for row, sensor in enumerate(sensors_to_plot):
    for col, machine in enumerate(sample_machines):
        machine_data = fd001_pdf[fd001_pdf['unit_id'] == machine].sort_values('cycle')
        axes[row][col].plot(machine_data['cycle'], machine_data[sensor], 
                           color='#1565C0', linewidth=1.2, alpha=0.8)
        axes[row][col].set_title(f"Machine {machine} — {sensor}")
        axes[row][col].set_xlabel("Cycle")
        axes[row][col].set_ylabel("Reading")
        
        axes[row][col].axvline(x=machine_data['cycle'].max(), 
                               color='red', linestyle='--', alpha=0.7, label='Failure')

plt.tight_layout()
plt.savefig("/Volumes/workspace/predictive_maintenance/raw_data/chart3_degradation_trends.png")
plt.show()
print(" Clear degradation trends visible toward failure point (red line)")

In [0]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("FD001 vs FD002 — Operating Conditions Comparison", 
             fontsize=13, fontweight='bold')

for i, setting in enumerate(['setting_1', 'setting_2', 'setting_3']):
    fd001_vals = pdf[pdf['source_dataset']=='FD001'][setting]
    fd002_vals = pdf[pdf['source_dataset']=='FD002'][setting]
    
    axes[i].boxplot([fd001_vals, fd002_vals], 
                    labels=['FD001', 'FD002'],
                    patch_artist=True,
                    boxprops=dict(facecolor='#90CAF9'),
                    medianprops=dict(color='#1565C0', linewidth=2))
    axes[i].set_title(f"{setting}")
    axes[i].set_ylabel("Value")

plt.tight_layout()
plt.savefig("/Volumes/workspace/predictive_maintenance/raw_data/chart4_operating_conditions.png")
plt.show()
print("FD002 shows wider spread — confirms multi-condition complexity")

In [0]:
pdf_rul = pdf.copy()
max_c = pdf_rul.groupby('unit_id')['cycle'].transform('max')
pdf_rul['RUL'] = max_c - pdf_rul['cycle']
pdf_rul['fail_30'] = (pdf_rul['RUL'] <= 30).astype(int)
pdf_rul['fail_15'] = (pdf_rul['RUL'] <= 15).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Class Balance — Failure Labels", fontsize=13, fontweight='bold')

for ax, col, title in zip(axes, ['fail_30', 'fail_15'], 
                           ['Failure in 30 Cycles', 'Failure in 15 Cycles']):
    counts = pdf_rul[col].value_counts()
    ax.pie(counts, labels=['No Failure', 'Failure'], 
           colors=['#90CAF9', '#EF5350'],
           autopct='%1.1f%%', startangle=90,
           wedgeprops={'edgecolor': 'white', 'linewidth': 2})
    ax.set_title(title)

plt.tight_layout()
plt.savefig("/Volumes/workspace/predictive_maintenance/raw_data/chart5_class_balance.png")
plt.show()

f30 = pdf_rul['fail_30'].mean() * 100
f15 = pdf_rul['fail_15'].mean() * 100
print(f"fail_30 class: {f30:.1f}% positive — manageable imbalance")
print(f"fail_15 class: {f15:.1f}% positive — more imbalanced, use F1 not accuracy")

In [0]:
import seaborn as sns

top7 = top_sensors  # from Cell 5

corr = pdf[top7].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues',
            ax=ax, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
ax.set_title("Top 7 Sensor Correlation Matrix", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("/Volumes/workspace/predictive_maintenance/raw_data/chart6_correlation.png")
plt.show()
print("✅ High correlation between some sensors — confirms redundancy in low-variance sensors")

In [0]:
sample = pdf_rul[pdf_rul['source_dataset']=='FD001'].sample(3000, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("RUL vs Key Sensor Readings — Degradation Signal", 
             fontsize=13, fontweight='bold')

for ax, sensor in zip(axes, top_sensors[:3]):
    ax.scatter(sample[sensor], sample['RUL'], 
               alpha=0.2, color='#1565C0', s=10)
    ax.set_xlabel(sensor)
    ax.set_ylabel("RUL (cycles remaining)")
    ax.set_title(f"RUL vs {sensor}")

plt.tight_layout()
plt.savefig("/Volumes/workspace/predictive_maintenance/raw_data/chart7_rul_vs_sensor.png")
plt.show()
print("Clear relationship between sensor readings and RUL — confirms predictive value")

In [0]:
# Normalize cycle position 0→1 for each machine
pdf_rul['lifecycle_pct'] = pdf_rul['cycle'] / max_c

# Bin lifecycle into 10 stages
pdf_rul['lifecycle_stage'] = pd.cut(pdf_rul['lifecycle_pct'], 
                                     bins=10, labels=False)

# Average sensor reading per stage
stage_avg = pdf_rul.groupby('lifecycle_stage')[top_sensors[:3]].mean()

fig, ax = plt.subplots(figsize=(12, 5))
for sensor in top_sensors[:3]:
    ax.plot(stage_avg.index, stage_avg[sensor], 
            marker='o', linewidth=2, label=sensor)

ax.set_title("Average Sensor Reading Across Machine Lifecycle\n(0 = New Machine, 9 = Near Failure)", 
             fontsize=13, fontweight='bold')
ax.set_xlabel("Lifecycle Stage (0=New → 9=Failure)")
ax.set_ylabel("Average Sensor Reading")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("/Volumes/workspace/predictive_maintenance/raw_data/chart8_lifecycle_health.png")
plt.show()
print("8 charts complete — EDA done!")

## Sensor Correlation Finding
All top 7 sensors show very high correlation (0.77–1.00).
This confirms that:
1. The 14 dropped sensors were correctly removed
2. Among top 7, some redundancy exists — rolling window 
   features will help differentiate their signals over time
3. We will keep all 7 for now and let XGBoost + SHAP 
   identify the truly important ones during training
   